In [1]:
# imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import ReduceLROnPlateau
# Use torch.amp directly for GradScaler
from torch.amp.grad_scaler import GradScaler
from torch.cuda.amp import autocast # Mixed Precision
from torchvision import transforms, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
)
from PIL import Image
import os
import random
import copy
import time
import logging # Logging
from tqdm import tqdm # Changed from tqdm.notebook

In [2]:
# --- Basic Logging Setup ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()], # Output to console
)

# --- Reproducibility ---
def set_seed(seed=42):
    """Sets the seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Ensure deterministic behavior for cuDNN
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    logging.info(f"Random seed set to {seed}")

SEED = 42
set_seed(SEED)

2025-05-05 21:28:14,131 [INFO] Random seed set to 42


In [3]:
# --- Configuration ---
# Path to your FER2013 CSV file
CSV_PATH = "fer2013.csv"
# Model/Checkpoint saving directory
MODEL_DIR = "models_checkpointed"
os.makedirs(MODEL_DIR, exist_ok=True)

# Training Hyperparameters
BATCH_SIZE = 16
EPOCHS_PHASE1 = 3 # Epochs for training the head only
EPOCHS_PHASE2 = 7 # Epochs for fine-tuning the whole model
TOTAL_EPOCHS = EPOCHS_PHASE1 + EPOCHS_PHASE2
K_FOLDS = 5
LEARNING_RATE_HEAD = 1e-3
LEARNING_RATE_BACKBONE = 1e-5
WEIGHT_DECAY = 1e-2
SCHEDULER_PATIENCE = 3 # LR scheduler patience
SCHEDULER_FACTOR = 0.1 # LR reduction factor
EARLY_STOPPING_PATIENCE = 5 # Early stopping patience based on F1
DROPOUT_RATE = 0.5 # As specified in methodology

# --- Load Data ---
logging.info(f"Loading data from {CSV_PATH}")
try:
    df = pd.read_csv(CSV_PATH)
except FileNotFoundError:
    logging.error(f"Error: CSV file not found at {CSV_PATH}")
    exit() # Exit if data file is missing

# Separate PrivateTest set
df_test = df[df["Usage"] == "PrivateTest"].copy()
df_train_val = df[df["Usage"].isin(["Training", "PublicTest"])].copy()
logging.info(f"Loaded {len(df_train_val)} samples for Training/Validation")
logging.info(f"Loaded {len(df_test)} samples for Final Testing")

# Map emotion: 3 (happy) -> 1, others -> 0
df_train_val["label"] = (df_train_val["emotion"] == 3).astype(int)
df_test["label"] = (df_test["emotion"] == 3).astype(int)

# Calculate Class Weights for the training/validation data
class_counts = df_train_val["label"].value_counts().sort_index()
if len(class_counts) < 2:
     logging.warning("Only one class found in training/validation data. Class weighting might not work as expected.")
     # Handle case with only one class if necessary, e.g., set weights to 1
     class_weights_tensor = torch.tensor([1.0, 1.0], dtype=torch.float32) # Default or adjust
else:
    total_samples = len(df_train_val)
    class_weights = total_samples / (len(class_counts) * class_counts)
    class_weights_tensor = torch.tensor(
        class_weights.values, dtype=torch.float32
    )
logging.info(f"Class counts (Train/Val): {class_counts.to_dict()}")
logging.info(f"Calculated class weights: {class_weights_tensor.numpy()}")

2025-05-05 21:28:14,140 [INFO] Loading data from fer2013.csv
2025-05-05 21:28:16,596 [INFO] Loaded 32298 samples for Training/Validation
2025-05-05 21:28:16,596 [INFO] Loaded 3589 samples for Final Testing
2025-05-05 21:28:16,599 [INFO] Class counts (Train/Val): {0: 24188, 1: 8110}
2025-05-05 21:28:16,600 [INFO] Calculated class weights: [0.6676451 1.9912454]


In [4]:
# --- Dataset Class ---
class FERDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            # Use np.fromiter which is generally safer/faster for text
            pixels = np.fromiter(
                row["pixels"].split(" "), dtype=np.uint8
            ).reshape(48, 48)
        except Exception as e:
            logging.error(f"Error processing row {idx}: {row['pixels'][:50]}...")
            # Optionally return a placeholder or skip this item
            # For now, re-raise the exception
            raise e
        img = Image.fromarray(pixels).convert("RGB")
        label = row["label"]
        if self.transform:
            img = self.transform(img)
        return img, label

# --- Transforms ---
# Added Data Augmentation for training
train_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

# Minimal transforms for validation/testing
val_test_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

In [5]:
# --- Model Definition ---
def get_model(dropout_rate=DROPOUT_RATE):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    num_ftrs = model.classifier[1].in_features
    # Add Dropout before the final layer as per methodology
    # Alternatively, could explore leveraging EfficientNet's built-in dropout
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout_rate, inplace=True),
        nn.Linear(num_ftrs, 2), # Binary classification
    )
    return model

In [6]:
# --- Training & Evaluation Functions ---
def train_one_epoch(
    model, loader, criterion, optimizer, device, scaler, phase="Head"
):
    model.train()
    running_loss = 0.0
    # Wrap loader with tqdm for progress bar
    pbar = tqdm(loader, desc=f"Training Epoch ({phase})", leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True) # More efficient zeroing

        # Mixed Precision Context
        # Use updated torch.amp.autocast syntax
        with torch.amp.autocast(device_type=device.type, enabled=(scaler is not None)): # <--- FIX HERE
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        if scaler: # If using mixed precision
            scaler.scale(loss).backward()
            # Unscales gradients and calls optimizer.step()
            scaler.step(optimizer)
            # Updates the scale for next iteration
            scaler.update()
        else: # Standard precision
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        pbar.set_postfix(loss=loss.item()) # Show current batch loss in progress bar

    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss


def evaluate(model, loader, criterion, device):
    model.eval()
    all_preds, all_targets = [], []
    all_probs = []
    running_loss = 0.0
    # Wrap loader with tqdm for progress bar
    pbar = tqdm(loader, desc="Evaluating", leave=False)
    with torch.no_grad():
        for imgs, labels in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            # Autocast can be used during inference as well, potentially faster
            # but usually not necessary unless memory is extremely tight.
            # with autocast(enabled=(device.type == 'cuda')):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)

            probabilities = torch.softmax(outputs, dim=1)[:, 1] # Prob of class 1

            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            all_probs.extend(probabilities.cpu().numpy())

    val_loss = running_loss / len(loader.dataset)
    # Ensure targets and preds are numpy arrays for sklearn metrics
    all_targets_np = np.array(all_targets)
    all_preds_np = np.array(all_preds)
    all_probs_np = np.array(all_probs)

    val_acc = accuracy_score(all_targets_np, all_preds_np)
    val_f1 = f1_score(all_targets_np, all_preds_np, average="binary", zero_division=0)
    val_prec = precision_score(all_targets_np, all_preds_np, average="binary", zero_division=0)
    val_rec = recall_score(all_targets_np, all_preds_np, average="binary", zero_division=0)
    # Handle case where only one class is present in targets during evaluation
    try:
        # Ensure there are samples for both classes for AUC calculation
        if len(np.unique(all_targets_np)) > 1:
             val_auc = roc_auc_score(all_targets_np, all_probs_np) # Binary AUC is standard
        else:
             val_auc = 0.0 # Or 0.5, or np.nan - depends on desired handling
             logging.warning("AUC calculation skipped: only one class present in evaluation targets.")
    except ValueError as e:
        val_auc = 0.0 # Or handle as appropriate
        logging.warning(f"AUC calculation failed: {e}")

    cm = confusion_matrix(all_targets_np, all_preds_np)
    report_dict = classification_report(
            all_targets_np, all_preds_np, target_names=["Not Happy", "Happy"], output_dict=True, zero_division=0
        )

    metrics = {
        "loss": val_loss,
        "accuracy": val_acc,
        "f1": val_f1,
        "precision": val_prec,
        "recall": val_rec,
        "auc": val_auc,
        "cm": cm,
        "report": report_dict,
        "preds": all_preds_np, # Return preds/targets if needed later
        "targets": all_targets_np
    }
    return metrics

In [7]:
# --- Main K-Fold Cross-Validation ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.info(f"Using device: {device}")
# Increase num_workers if possible, handle Windows case
num_workers = 4 if os.name != "nt" and torch.cuda.is_available() else 0
logging.info(f"Using {num_workers} workers for DataLoaders")

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
X = df_train_val.index.values # Use index for splitting
y = df_train_val["label"].values

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    logging.info(f"--- Starting Fold {fold+1}/{K_FOLDS} ---")
    fold_start_time = time.time()

    train_df = df_train_val.iloc[train_idx]
    val_df = df_train_val.iloc[val_idx]

    train_ds = FERDataset(train_df, transform=train_transform)
    val_ds = FERDataset(val_df, transform=val_test_transform)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True
    )

    model = get_model().to(device)
    # Use class weights in criterion
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))

    # Mixed Precision Scaler (only if CUDA is available)
    # Use updated initialization
    scaler = torch.amp.GradScaler('cuda') if device.type == "cuda" else None
    if scaler:
        logging.info("Using Mixed Precision (AMP) with torch.amp.GradScaler")

    # --- Phase 1: Train the Head ---
    logging.info("--- Phase 1: Training Head ---")
    # Freeze backbone
    for param in model.features.parameters():
        param.requires_grad = False
    # Ensure classifier parameters require grad
    for param in model.classifier.parameters():
        param.requires_grad = True

    # Optimizer for head only
    optimizer = optim.AdamW(
        # Filter parameters that require gradients
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE_HEAD,
        weight_decay=WEIGHT_DECAY
    )

    for epoch in range(EPOCHS_PHASE1):
        epoch_start_time = time.time()
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, device, scaler, phase="Head"
        )
        epoch_time = time.time() - epoch_start_time
        logging.info(
            f"Fold {fold+1} Phase 1 - Epoch {epoch+1}/{EPOCHS_PHASE1}, Train Loss: {train_loss:.4f}, Time: {epoch_time:.2f}s"
        )

    # --- Phase 2: Fine-tune the whole model ---
    logging.info("--- Phase 2: Fine-tuning Full Model ---")
    # Unfreeze backbone
    for param in model.features.parameters():
        param.requires_grad = True

    # Optimizer with differential learning rates
    optimizer = optim.AdamW(
        [
            {
                "params": model.features.parameters(),
                "lr": LEARNING_RATE_BACKBONE,
            },
            {
                "params": model.classifier.parameters(),
                "lr": LEARNING_RATE_HEAD, # Keep head LR higher initially
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )

    # Learning Rate Scheduler
    scheduler = ReduceLROnPlateau(
        optimizer, mode='min', factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE, verbose=False # Quieter scheduler
    )

    # Early Stopping variables
    best_val_f1 = -1.0 # Initialize to handle F1=0 case
    patience_counter = 0
    best_epoch = -1
    checkpoint_path = os.path.join(MODEL_DIR, f"best_model_fold_{fold+1}.pth")

    for epoch in range(EPOCHS_PHASE2):
        epoch_start_time = time.time()
        current_epoch_total = EPOCHS_PHASE1 + epoch + 1

        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, device, scaler, phase="Full"
        )
        val_metrics = evaluate(model, val_loader, criterion, device)

        epoch_time = time.time() - epoch_start_time
        # Log current learning rate(s)
        current_lrs = [group['lr'] for group in optimizer.param_groups]
        logging.info(
            f"Fold {fold+1} Phase 2 - Epoch {epoch+1}/{EPOCHS_PHASE2} (Total: {current_epoch_total}), "
            f"LR: {current_lrs}, Train Loss: {train_loss:.4f}, Val Loss: {val_metrics['loss']:.4f}, "
            f"Val Acc: {val_metrics['accuracy']:.4f}, Val F1: {val_metrics['f1']:.4f}, "
            # f"Val Prec: {val_metrics['precision']:.4f}, Val Rec: {val_metrics['recall']:.4f}, " # Optional: reduce log verbosity
            f"Val AUC: {val_metrics['auc']:.4f}, Time: {epoch_time:.2f}s"
        )

        # Step the scheduler based on validation loss
        scheduler.step(val_metrics['loss'])

        # Early Stopping Check based on F1 score
        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            patience_counter = 0
            best_epoch = current_epoch_total
            # Save checkpoint (model, optimizer, epoch)
            try:
                torch.save({
                    'epoch': current_epoch_total,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(), # Save scheduler state too
                    'loss': val_metrics['loss'],
                    'f1': best_val_f1,
                }, checkpoint_path)
                logging.info(
                    f"  -> New best F1: {best_val_f1:.4f} at epoch {best_epoch}. Checkpoint saved to {checkpoint_path}"
                )
            except Exception as e:
                 logging.error(f"Error saving checkpoint: {e}")
        else:
            patience_counter += 1
            logging.info(f"  -> F1 did not improve. Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            logging.info(f"  -> Early stopping triggered at epoch {current_epoch_total}.")
            break

    # Load best model state for final evaluation for this fold
    if os.path.exists(checkpoint_path):
        logging.info(f"Loading best model from {checkpoint_path} (Epoch {best_epoch}, F1: {best_val_f1:.4f})")
        try:
            checkpoint = torch.load(checkpoint_path, map_location=device) # Load to correct device
            model.load_state_dict(checkpoint['model_state_dict'])
            # Optionally load optimizer and scheduler if needed for resuming, but not for final eval
        except Exception as e:
            logging.error(f"Error loading checkpoint: {e}. Using last model state.")
            # Fallback to using the model state from the last epoch if loading fails
    else:
        logging.warning("No best model checkpoint found for this fold. Using last model state.")

    # Final Validation for the fold using the best loaded model
    logging.info(f"--- Evaluating Best Model for Fold {fold+1} ---")
    final_fold_metrics = evaluate(model, val_loader, criterion, device)

    logging.info(f"Fold {fold+1} Final Validation Results (Best Model):")
    logging.info(f"  Accuracy:  {final_fold_metrics['accuracy']:.4f}")
    logging.info(f"  F1 Score:  {final_fold_metrics['f1']:.4f}")
    logging.info(f"  Precision: {final_fold_metrics['precision']:.4f}")
    logging.info(f"  Recall:    {final_fold_metrics['recall']:.4f}")
    logging.info(f"  AUC:       {final_fold_metrics['auc']:.4f}")
    logging.info(f"  Loss:      {final_fold_metrics['loss']:.4f}")
    logging.info("  Confusion Matrix:")
    logging.info(f"\n{final_fold_metrics['cm']}")

    fold_results.append(final_fold_metrics)
    fold_time = time.time() - fold_start_time
    logging.info(f"--- Fold {fold+1} completed in {fold_time:.2f}s ---")

# --- Cross-Validation Summary ---
logging.info("--- Cross-Validation Summary ---")
if fold_results: # Check if any folds completed
    accs = [r["accuracy"] for r in fold_results]
    f1s = [r["f1"] for r in fold_results]
    precs = [r["precision"] for r in fold_results]
    recs = [r["recall"] for r in fold_results]
    aucs = [r["auc"] for r in fold_results]

    logging.info(f"Mean Accuracy:  {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    logging.info(f"Mean F1 Score:  {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    logging.info(f"Mean Precision: {np.mean(precs):.4f} ± {np.std(precs):.4f}")
    logging.info(f"Mean Recall:    {np.mean(recs):.4f} ± {np.std(recs):.4f}")
    logging.info(f"Mean AUC:       {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
else:
    logging.warning("No fold results to summarize.")



2025-05-05 21:28:16,671 [INFO] Using device: cuda
2025-05-05 21:28:16,671 [INFO] Using 4 workers for DataLoaders
2025-05-05 21:28:16,678 [INFO] --- Starting Fold 1/5 ---
2025-05-05 21:28:17,174 [INFO] Using Mixed Precision (AMP) with torch.amp.GradScaler
2025-05-05 21:28:17,174 [INFO] --- Phase 1: Training Head ---
Training Epoch (Head):   0%|          | 0/1615 [00:00<?, ?it/s]/home/marvin/Developer/NoSmiles/local/.venv/lib/python3.12/site-packages/torch/nn/modules/linear.py:125: UserWarning: Attempting to use hipBLASLt on an unsupported architecture! Overriding blas backend to hipblas (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:310.)
  return F.linear(input, self.weight, self.bias)
2025-05-05 21:29:46,719 [INFO] Fold 1 Phase 1 - Epoch 1/3, Train Loss: 0.6199, Time: 89.54s
2025-05-05 21:31:16,007 [INFO] Fold 1 Phase 1 - Epoch 2/3, Train Loss: 0.6146, Time: 89.29s
2025-05-05 21:32:45,890 [INFO] Fold 1 Phase 1 - Epoch 3/3, Train Loss: 0.6100, Time: 89.88s
2025-05-05 21:32

NameError: name 'predicted' is not defined

In [ ]:
# --- Training Final Model on Full Train/Val Data ---
logging.info("--- Training Final Model on Full Train/Val Dataset ---")
final_model = get_model().to(device)
final_ds = FERDataset(df_train_val, transform=train_transform) # Use train transforms
final_loader = DataLoader(
    final_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=True
)
criterion_final = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
scaler_final = torch.amp.GradScaler('cuda') if device.type == "cuda" else None

# --- Final Model - Phase 1: Train the Head ---
logging.info("--- Final Model - Phase 1: Training Head ---")
for param in final_model.features.parameters():
    param.requires_grad = False
for param in final_model.classifier.parameters():
        param.requires_grad = True
optimizer_head_final = optim.AdamW(
    filter(lambda p: p.requires_grad, final_model.parameters()),
    lr=LEARNING_RATE_HEAD,
    weight_decay=WEIGHT_DECAY
)
for epoch in range(EPOCHS_PHASE1):
    epoch_start_time = time.time()
    train_loss = train_one_epoch(
        final_model, final_loader, criterion_final, optimizer_head_final, device, scaler_final, phase="Head"
    )
    epoch_time = time.time() - epoch_start_time
    logging.info(
        f"Final Phase 1 - Epoch {epoch+1}/{EPOCHS_PHASE1}, Train Loss: {train_loss:.4f}, Time: {epoch_time:.2f}s"
    )

# --- Final Model - Phase 2: Fine-tune the whole model ---
logging.info("--- Final Model - Phase 2: Fine-tuning Full Model ---")
for param in final_model.features.parameters():
    param.requires_grad = True
optimizer_full_final = optim.AdamW(
    [
        {
            "params": final_model.features.parameters(),
            "lr": LEARNING_RATE_BACKBONE,
        },
        {
            "params": final_model.classifier.parameters(),
            "lr": LEARNING_RATE_HEAD,
        },
    ],
    weight_decay=WEIGHT_DECAY,
)
# No scheduler or early stopping needed for final training run
for epoch in range(EPOCHS_PHASE2):
    epoch_start_time = time.time()
    train_loss = train_one_epoch(
        final_model, final_loader, criterion_final, optimizer_full_final, device, scaler_final, phase="Full"
    )
    epoch_time = time.time() - epoch_start_time
    logging.info(
        f"Final Phase 2 - Epoch {epoch+1}/{EPOCHS_PHASE2}, Train Loss: {train_loss:.4f}, Time: {epoch_time:.2f}s"
    )

In [ ]:
# Save final model (only state_dict needed for inference)
final_model_save_path = os.path.join(MODEL_DIR, "efficientnetb0_fer2013_happy_final.pth")
try:
    torch.save(final_model.state_dict(), final_model_save_path)
    logging.info(f"Final model state_dict saved to {final_model_save_path}")
except Exception as e:
    logging.error(f"Error saving final model: {e}")

In [ ]:
# --- Final Evaluation on Test Set ---
logging.info("--- Evaluating Final Model on PrivateTest Set ---")
if len(df_test) > 0:
    test_ds = FERDataset(df_test, transform=val_test_transform)
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True
    )

    # Load the final trained model
    try:
        final_model.load_state_dict(torch.load(final_model_save_path, map_location=device))
        final_model.to(device) # Ensure model is on correct device
        logging.info(f"Successfully loaded final model from {final_model_save_path}")

        # Evaluate on the test set
        test_metrics = evaluate(final_model, test_loader, criterion_final, device) # Use same criterion setup

        logging.info("Final Test Set Results:")
        logging.info(f"  Accuracy:  {test_metrics['accuracy']:.4f}")
        logging.info(f"  F1 Score:  {test_metrics['f1']:.4f}")
        logging.info(f"  Precision: {test_metrics['precision']:.4f}")
        logging.info(f"  Recall:    {test_metrics['recall']:.4f}")
        logging.info(f"  AUC:       {test_metrics['auc']:.4f}")
        logging.info(f"  Loss:      {test_metrics['loss']:.4f}")
        logging.info("  Confusion Matrix:")
        logging.info(f"\n{test_metrics['cm']}")
        logging.info("  Classification Report:")
        # Use the returned targets/preds from evaluate for the report
        logging.info(f"\n{classification_report(test_metrics['targets'], test_metrics['preds'], target_names=['Not Happy', 'Happy'], zero_division=0)}")

    except FileNotFoundError:
        logging.error(f"Final model file not found at {final_model_save_path}. Cannot evaluate on test set.")
    except Exception as e:
        logging.error(f"Error during final evaluation on test set: {e}")

else:
    logging.warning("PrivateTest set is empty or could not be loaded. Skipping final evaluation.")


# --- Optimization Notes ---
logging.info("--- Optimization Notes ---")
logging.info("The saved final model is a standard PyTorch state dictionary.")
logging.info("For near real-time performance as discussed in the methodology,")
logging.info("further optimization steps like conversion to ONNX, TensorRT, or OpenVINO,")
logging.info("and potentially quantization, would be necessary after training.")
